# Notebook 19 — TriMVPP interpretability rerun

Notebook này không train lại model. Nó dùng 5 checkpoint Cross-Attention đã chốt (seed 5, 6, 7, 37, 42), chỉ trên Test, ClinVarHQ và ProteinGym; không chạy UniProt.

Thứ tự chạy: **A predictions toàn bộ mẫu → D can thiệp AF toàn bộ mẫu → B Integrated Gradients (IG) theo AF strata → kiểm tra độ ổn định nền IG → C occlusion cá thể và theo nhóm → E attention maps**. Mỗi giai đoạn ở cell riêng, ghi các Parquet chunk để resume. Cell C và E dừng nếu kiểm tra nền IG chưa đạt Spearman ≥ 0.8; chỉ bật override sau khi đã xem bảng độ ổn định.

Kết quả v2 nằm trong `experiments/cross_attention_explainability_v2`; output v1 được giữ nguyên. Các giá trị attention/attribution mô tả hành vi của model, không chứng minh quan hệ nhân quả sinh học.

In [1]:
from pathlib import Path
import gc
import hashlib
import importlib
import json
import os
import re
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "core").is_dir() and (path / "controllers").is_dir()),
    Path.cwd(),
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

BASE_DIR = Path("F:/variant_data")
SOURCE_BATCH_DIR = REPO_ROOT / "experiments" / "fm_ablation_20261117_003149"
REFERENCE_BATCH_DIR = REPO_ROOT / "experiments" / "fusion_classifier_benchmark_20261120_192605"
SOURCE_RUNS_ROOT = (
    SOURCE_BATCH_DIR / "module06_architecture_experiments"
    / "step2_sequence_length_ablation" / "runs"
)
TOKEN_CACHE_INDEX_PATH = (
    REFERENCE_BATCH_DIR / "module06_architecture_experiments"
    / "fm_ablation_token_cache_index.json"
)
NORMALIZATION_ARTIFACTS_DIR = BASE_DIR / "normalization_artifacts"
OUTPUT_ROOT = REPO_ROOT / "experiments" / "cross_attention_explainability_v2"

DNA_MODEL_NAME = "nt_v1_500m_dna_1001"
PROTEIN_MODEL_NAME = "esm1b_650m_prot_501"
GEOMETRY_ROOT_NAME = "geometry_v2_k64_step2_matched"
GEOMETRY_ROOT = BASE_DIR / GEOMETRY_ROOT_NAME / "{context}"
FM_PROFILE_JSON = BASE_DIR / "profiling" / "fm_profiling_sequence_ablation.json"

SEEDS_TO_EXPLAIN = [5, 6, 7, 37, 42]
DATASETS_TO_EXPLAIN = {
    "test": "Train3Val_Test",
    "clinvarhq": "Train3Val_ClinVarHQ",
    "proteingym": "Train3Val_ProteinGym",
}
EXPECTED_TEST_MCC = {5: 0.8381, 6: 0.8316, 7: 0.8395, 37: 0.8424, 42: 0.8345}
MCC_REFERENCE_TOLERANCE = 0.002
SAMPLES_PER_LABEL_AF_STRATUM = 25
ATTENTION_SAMPLES_PER_LABEL = 24
SENSITIVITY_SAMPLES_PER_LABEL = 10
BACKGROUND_SAMPLES = 256
BACKGROUND_SEEDS = (2026, 2027, 2028)
IG_STEPS = 64
IG_RETRY_STEPS = 128
IG_INTERNAL_BATCH_SIZE = 8
IG_CHUNK_SIZE = 5
SENSITIVITY_CHUNK_SIZE = 4
OCCLUSION_CHUNK_SIZE = 16
ATTENTION_CHUNK_SIZE = 4
ALLOW_LOW_BACKGROUND_STABILITY_TO_CONTINUE = False
SHAP_SAMPLES_PER_LABEL = 5
SHAP_BACKGROUND_SAMPLES = 32
SHAP_BACKGROUND_SEED = 2030
SHAP_GRADIENT_NSAMPLES = 128
SHAP_GRADIENT_BATCH_SIZE = 16
SHAP_CHUNK_SIZE = 4
SHAP_DEEP_ADDITIVITY_FALLBACK = True

BATCH_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_CONFIG = {
    "attention_dropout": 0.18776012701077605,
    "dropout": 0.1955626033738947,
    "early_stop_patience": 6,
    "feature_dropout": 0.2507543060224074,
    "ffn_multiplier": 4,
    "k_radius": 16,
    "learning_rate": 0.00012206472463506827,
    "lr_factor": 0.5,
    "lr_patience": 3,
    "max_epochs": 30,
    "model_dim": 512,
    "num_heads": 8,
    "num_layers": 3,
    "pooling": "mean",
    "weight_decay": 3.380881049679085e-06,
}
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE, "| output:", OUTPUT_ROOT)

Device: cuda | output: d:\Missense_variant_pathogenicity_prediction\experiments\cross_attention_explainability_v2


In [2]:
from core.module05v2_fusion_classifier.pipeline import FusionBatchPipeline
from core.module05v2_fusion_classifier.dataset import BIO_CORE_COLUMNS, SPLICEAI_COLUMNS
from core.module06_architecture_experiments.fm_ablation_preparation import (
    build_model_configs, build_split_configs,
)
from core.module06_architecture_experiments.cross_attention_ablation_studies import GEOM_FEATURES
from core.module06_architecture_experiments.architecture_experiment_runner import (
    _architecture_loader, _attach_token_cache,
)
from core.module06_architecture_experiments.cross_attention_3modal_experiment import (
    _builder as build_cross_attention_model,
)
from core.module06_architecture_experiments.cross_attention_explainability import (
    empirical_baseline, select_stratified_indices, move_batch_to_device,
)
from core.module06_architecture_experiments.cross_attention_interpretability_v2 import (
    build_af_stratified_manifest, select_spread_subset, PartitionedParquetStore,
    CrossAttentionInterpretabilityExperiment,
)

if not TOKEN_CACHE_INDEX_PATH.is_file():
    raise FileNotFoundError(f"Notebook 17 token cache index is missing: {TOKEN_CACHE_INDEX_PATH}")
TOKEN_CACHE_INDEX = json.loads(TOKEN_CACHE_INDEX_PATH.read_text(encoding="utf-8"))
DNA_MODEL, PROTEIN_MODEL = [
    next(model for model in build_model_configs(1001, 501) if model["name"] == name)
    for name in (DNA_MODEL_NAME, PROTEIN_MODEL_NAME)
]
SPLITS = build_split_configs(BASE_DIR)
DATASETS = [
    {"name": "Train3Val_Test", "train": "train3", "val": "val", "test": "test"},
    {"name": "Train3Val_ClinVarHQ", "train": "train3", "val": "val", "test": "clinvarhq"},
    {"name": "Train3Val_ProteinGym", "train": "train3", "val": "val", "test": "proteingym"},
]
BIO_FEATURE_NAMES = [*BIO_CORE_COLUMNS, *SPLICEAI_COLUMNS]
TARGET = {
    "name": "B_with_geometry",
    "ablation": "bio_core_dna_prot_spliceai_geom",
    "active_modalities": ["dna", "prot", "bio_core", "spliceai", "geom"],
    "bio_feature_names": BIO_FEATURE_NAMES,
    "geom_feature_names": list(GEOM_FEATURES),
    "geometry_dna_name": DNA_MODEL_NAME,
    "geometry_protein_name": PROTEIN_MODEL_NAME,
}
PIPELINE_CONFIG = {
    "batch_size": BATCH_SIZE, "epochs": MODEL_CONFIG["max_epochs"],
    "lr": MODEL_CONFIG["learning_rate"], "weight_decay": MODEL_CONFIG["weight_decay"],
    "lr_factor": MODEL_CONFIG["lr_factor"], "lr_patience": MODEL_CONFIG["lr_patience"],
    "early_stop_patience": MODEL_CONFIG["early_stop_patience"],
    "num_workers": 0, "seed": SEEDS_TO_EXPLAIN[0], "xgb_n_jobs": 1,
}
pipeline = FusionBatchPipeline(
    base_dir=str(BASE_DIR), config=PIPELINE_CONFIG, datasets=DATASETS,
    models_space=[DNA_MODEL, PROTEIN_MODEL], pooling_strategies=["center"],
    experiments=[{"name": "CrossAttentionExplainability", "type": "pytorch", "fusion": "cross_attention"}],
    explainability={"enable_shap": False, "enable_lime": False},
    fm_profile_json=str(FM_PROFILE_JSON), geom_profile_json=None,
    geometry_dir=str(GEOMETRY_ROOT), geom_feature_names=GEOM_FEATURES,
    batch_run_dir=str(OUTPUT_ROOT),
    embedding_dir=str(BASE_DIR / "fm_embeddings" / "sequence_ablation"),
)
CONTEXT = {
    "pipeline": pipeline, "TARGET_CONFIGS": [TARGET], "datasets": DATASETS,
    "DNA_MODEL": DNA_MODEL, "PROTEIN_MODEL": PROTEIN_MODEL, "DEVICE": DEVICE,
    "token_embedding_cache_index": TOKEN_CACHE_INDEX,
    "token_batch_size": BATCH_SIZE, "token_only": True,
}
print("DNA/Protein:", DNA_MODEL_NAME, "+", PROTEIN_MODEL_NAME)
print("Features:", len(BIO_FEATURE_NAMES), "bio/SpliceAI +", len(GEOM_FEATURES), "geometry")
from core.module06_architecture_experiments.cross_attention_shap import (
    CrossAttentionShapExperiment,
)


c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict_clean\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DNA/Protein: nt_v1_500m_dna_1001 + esm1b_650m_prot_501
Features: 20 bio/SpliceAI + 8 geometry


In [3]:
SOURCE_INFO = {}
audit_rows = []
EXPECTED_ARCHITECTURE = "cross_attention_3modal_ref_alt_concat_window33_sequence_length_ablation"
EXPECTED_VARIANT = "ref_alt_concat_window_33_sequence_length_ablation"

for seed in SEEDS_TO_EXPLAIN:
    run_name = f"step2_nt_v1_dna1001_esm1b_prot501_seed{seed}"
    run_dir = SOURCE_RUNS_ROOT / run_name
    checkpoint = run_dir / "checkpoints" / "B_with_geometry" / "best_model.pth"
    manifest_path = run_dir / "_resume_manifest.json"
    validation_paths = list(run_dir.glob("*_validation_metrics.csv"))
    if not checkpoint.is_file() or not manifest_path.is_file() or not validation_paths:
        raise FileNotFoundError(f"Incomplete source checkpoint bundle for seed {seed}: {run_dir}")

    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    saved = manifest.get("config", {})
    mismatches = []
    if saved.get("architecture") != EXPECTED_ARCHITECTURE:
        mismatches.append("architecture")
    if saved.get("variant") != EXPECTED_VARIANT:
        mismatches.append("variant")
    if saved.get("dna_model", {}).get("name") != DNA_MODEL_NAME:
        mismatches.append("DNA model")
    if saved.get("protein_model", {}).get("name") != PROTEIN_MODEL_NAME:
        mismatches.append("protein model")
    signature = saved.get("input_data_signature", {})
    if signature.get("geometry_root") != GEOMETRY_ROOT_NAME:
        mismatches.append("geometry root")
    if signature.get("token_window") != "variant-centered ref/alt, k_radius=16, length=33":
        mismatches.append("token window")
    config_keys = [key for key in saved if key.startswith(run_name + "_config")]
    if len(config_keys) != 1:
        mismatches.append("seed-specific model config")
    else:
        source_model_config = saved[config_keys[0]]
        model_keys = ("attention_dropout", "dropout", "feature_dropout",
                      "ffn_multiplier", "k_radius", "model_dim",
                      "num_heads", "num_layers", "pooling")
        if any(source_model_config.get(key) != MODEL_CONFIG[key] for key in model_keys):
            mismatches.append("model hyperparameters")
    saved_target = saved.get("target_configs", [{}])[0]
    if saved_target.get("name") != "B_with_geometry":
        mismatches.append("target config")
    if saved_target.get("geom_feature_names") != list(GEOM_FEATURES):
        mismatches.append("geometry feature set")

    validation = pd.read_csv(validation_paths[0])
    selected = validation[
        validation["Target_Config"].astype(str).eq("B_with_geometry")
        & pd.to_numeric(validation["Seed"], errors="coerce").eq(int(seed))
    ]
    if selected.empty:
        raise ValueError(f"No validation record for seed {seed} in {validation_paths[0]}")
    row = selected.iloc[0]
    if mismatches:
        raise ValueError(f"Seed {seed} checkpoint does not match this notebook: {mismatches}")
    SOURCE_INFO[int(seed)] = {
        "checkpoint": str(checkpoint.resolve()),
        "validation_csv": str(validation_paths[0].resolve()),
        "best_epoch": int(float(row.get("Best_Epoch", 1))),
        "val_mcc": float(row["Val_MCC"]),
        "threshold": float(row["Decision_Threshold"]),
    }
    audit_rows.append({
        "Seed": int(seed),
        "Best_Epoch": SOURCE_INFO[int(seed)]["best_epoch"],
        "Val_MCC": SOURCE_INFO[int(seed)]["val_mcc"],
        "Threshold": SOURCE_INFO[int(seed)]["threshold"],
        "Checkpoint": str(checkpoint),
        "Status": "ready",
    })

CHECKPOINT_AUDIT = pd.DataFrame(audit_rows)
display(CHECKPOINT_AUDIT)

,Seed,Best_Epoch,Val_MCC,Threshold,Checkpoint,Status
0,5,2,0.8209,0.888212,d:\Missense_variant_pathogenicity_prediction\e...,ready
1,6,4,0.8238,0.918221,d:\Missense_variant_pathogenicity_prediction\e...,ready
2,7,2,0.8235,0.559536,d:\Missense_variant_pathogenicity_prediction\e...,ready
3,37,2,0.8243,0.479132,d:\Missense_variant_pathogenicity_prediction\e...,ready
4,42,2,0.8210,0.814246,d:\Missense_variant_pathogenicity_prediction\e...,ready


In [4]:
def make_explanation_loader(split_name, seed):
    base_loader = _architecture_loader(
        pipeline=pipeline,
        context=CONTEXT,
        target=TARGET,
        split_name=split_name,
        pooling="center",
        dna_model=DNA_MODEL,
        protein_model=PROTEIN_MODEL,
        is_train=False,
        geometry_context="train3",
        seed=int(seed),
        embedding_representation="ref_alt_concat",
    )
    return _attach_token_cache(
        base_loader,
        CONTEXT,
        split_name,
        is_train=False,
        window_radius=int(MODEL_CONFIG["k_radius"]),
    )


def load_seed_model(seed, dataset):
    dimensions = {
        "token_dna": int(dataset.token_hidden_sizes["dna"]),
        "token_protein": int(dataset.token_hidden_sizes["protein"]),
        "bio": int(dataset.bio_tensor.shape[1]),
        "geometry": int(dataset.geom_tensor.shape[1]),
    }
    model = build_cross_attention_model(dimensions, MODEL_CONFIG).to(DEVICE)
    state = torch.load(
        SOURCE_INFO[int(seed)]["checkpoint"], map_location=DEVICE, weights_only=True
    )
    model.load_state_dict(state)
    model.eval()
    return model


print("Loader and five fixed checkpoints are ready.")


Loader and five fixed checkpoints are ready.


## Backgrounds and AF strata

IG uses 256 class-stratified train3 examples for each background seed (2026, 2027, 2028). AF strata are built from raw gnomAD exome AF in the original VEP parquet, before normalization. The selected examples and their sampling weights are saved to `sample_manifest.csv`.

In [5]:
TRAIN_LOADER = make_explanation_loader("train3", SEEDS_TO_EXPLAIN[0])
TRAIN_DATASET = TRAIN_LOADER.dataset
BACKGROUND_BASELINES = {}
for background_seed in BACKGROUND_SEEDS:
    background_indices = select_stratified_indices(
        TRAIN_DATASET, BACKGROUND_SAMPLES, seed=background_seed
    )
    background_batch = TRAIN_DATASET[background_indices]
    BACKGROUND_BASELINES[int(background_seed)] = empirical_baseline(
        background_batch, background_batch
    )
    print(f"Background {background_seed}: {len(background_indices)} train3 examples")
    del background_batch

TRAIN_FEATURE_MEDIANS = {
    "bio_features": TRAIN_DATASET.bio_tensor.median(dim=0).values.cpu(),
    "geom_features": TRAIN_DATASET.geom_tensor.median(dim=0).values.cpu(),
}

# Module 4 transforms AF as log10(raw_AF + 1e-8), then applies its train-fitted scaler.
frequency_scaler = joblib.load(NORMALIZATION_ARTIFACTS_DIR / "bio_freq_scaler.pkl")
zero_log_af = np.log10(np.full((1, 2), 1e-8, dtype=np.float64))
AF_ZERO_VALUES = frequency_scaler.transform(zero_log_af).reshape(-1).astype(np.float32)
af_indices = [BIO_FEATURE_NAMES.index(name) for name in ("AF", "gnomADe_AF")]
AF_MEDIAN_VALUES = TRAIN_FEATURE_MEDIANS["bio_features"][af_indices].numpy().astype(np.float32)
print("Normalized AF values for raw zero [AF, gnomADe_AF]:", AF_ZERO_VALUES)
print("Train3 normalized AF medians [AF, gnomADe_AF]:", AF_MEDIAN_VALUES)

af_manifests = []
for split_name, dataset_name in DATASETS_TO_EXPLAIN.items():
    raw_path = BASE_DIR / f"{split_name}_full_seq_after_vep_final.parquet"
    if not raw_path.is_file():
        raise FileNotFoundError(f"Raw AF metadata parquet not found: {raw_path}")
    raw_meta = pd.read_parquet(raw_path, columns=["Variant_ID", "gnomADe_AF"])
    loader = make_explanation_loader(split_name, SEEDS_TO_EXPLAIN[0])
    manifest = build_af_stratified_manifest(
        loader.dataset.df, raw_meta, dataset_name,
        per_cell=SAMPLES_PER_LABEL_AF_STRATUM, seed=2026,
    )
    print(dataset_name, "n=", len(loader.dataset), "selected=", len(manifest))
    display(manifest.groupby(["True_Label", "AF_Stratum"], observed=True).size().rename("N_Selected").reset_index())
    af_manifests.append(manifest)
    del loader, raw_meta, manifest
    gc.collect()
SAMPLE_MANIFEST = pd.concat(af_manifests, ignore_index=True)
ATTENTION_MANIFEST = select_spread_subset(
    SAMPLE_MANIFEST, per_label=ATTENTION_SAMPLES_PER_LABEL, seed=2026, attention=True
)
SENSITIVITY_MANIFEST = select_spread_subset(
    SAMPLE_MANIFEST, per_label=SENSITIVITY_SAMPLES_PER_LABEL, seed=2026
)
SHAP_MANIFEST = select_spread_subset(
    SAMPLE_MANIFEST, per_label=SHAP_SAMPLES_PER_LABEL, seed=SHAP_BACKGROUND_SEED
)
shap_background_indices = select_stratified_indices(
    TRAIN_DATASET, SHAP_BACKGROUND_SAMPLES, seed=SHAP_BACKGROUND_SEED
)
shap_background_batch = TRAIN_DATASET[shap_background_indices]
SHAP_BACKGROUND_INPUTS = {
    key: shap_background_batch[key].to(torch.float32).cpu()
    for key in ("token_dna", "token_protein", "bio_features", "geom_features")
}
train_label_column = next(
    name for name in ("Pathogenicity_Label", "Label", "label", "target", "Target")
    if name in TRAIN_DATASET.df.columns
)
SHAP_BACKGROUND_MANIFEST = TRAIN_DATASET.df.iloc[shap_background_indices][
    ["Variant_ID", train_label_column]
].reset_index(drop=True).rename(columns={train_label_column: "True_Label"})
SHAP_BACKGROUND_MANIFEST.insert(0, "Sample_Index", shap_background_indices)
SHAP_BACKGROUND_MANIFEST["Background_Seed"] = SHAP_BACKGROUND_SEED
SHAP_BACKGROUND_SIGNATURE = {"samples": SHAP_BACKGROUND_SAMPLES, "seed": SHAP_BACKGROUND_SEED}
del shap_background_batch
attention_ids = set(ATTENTION_MANIFEST["Variant_ID"].astype(str))
SAMPLE_MANIFEST["For_Attention"] = SAMPLE_MANIFEST["Variant_ID"].astype(str).isin(attention_ids)

# Fail before replacing provenance files if an existing v2 run used another sample list.
existing_signature_path = OUTPUT_ROOT / "run_signature.json"
if existing_signature_path.is_file():
    existing_signature = json.loads(existing_signature_path.read_text(encoding="utf-8"))
    candidate_hashes = {
        "sample_manifest_sha256": hashlib.sha256(
            SAMPLE_MANIFEST.to_csv(index=False).encode("utf-8")
        ).hexdigest(),
        "attention_manifest_sha256": hashlib.sha256(
            ATTENTION_MANIFEST.to_csv(index=False).encode("utf-8")
        ).hexdigest(),
        "sensitivity_manifest_sha256": hashlib.sha256(
            SENSITIVITY_MANIFEST.to_csv(index=False).encode("utf-8")
        ).hexdigest(),
    }
    changed = [key for key, value in candidate_hashes.items()
               if existing_signature.get(key) != value]
    if changed:
        raise ValueError(
            f"Sample manifest differs from existing v2 results ({changed}). "
            "Change OUTPUT_ROOT before continuing."
        )

# Refuse to overwrite SHAP provenance if the requested subset/background changed.
existing_shap_signature_path = OUTPUT_ROOT / "shap_run_signature.json"
if existing_shap_signature_path.is_file():
    previous_shap_signature = json.loads(
        existing_shap_signature_path.read_text(encoding="utf-8")
    )
    candidate_shap_manifest_hash = hashlib.sha256(
        SHAP_MANIFEST.to_csv(index=False).encode("utf-8")
    ).hexdigest()
    candidate_shap_background_hash = hashlib.sha256(
        SHAP_BACKGROUND_MANIFEST.to_csv(index=False).encode("utf-8")
    ).hexdigest()
    if (
        previous_shap_signature.get("sample_manifest_sha256") != candidate_shap_manifest_hash
        or previous_shap_signature.get("background_manifest_sha256") != candidate_shap_background_hash
    ):
        raise ValueError("SHAP sample/background manifest changed; use a new OUTPUT_ROOT.")

SAMPLE_MANIFEST.to_csv(OUTPUT_ROOT / "sample_manifest.csv", index=False)
ATTENTION_MANIFEST.to_csv(OUTPUT_ROOT / "attention_sample_manifest.csv", index=False)
SENSITIVITY_MANIFEST.to_csv(OUTPUT_ROOT / "ig_sensitivity_sample_manifest.csv", index=False)
SHAP_MANIFEST.to_csv(OUTPUT_ROOT / "shap_sample_manifest.csv", index=False)
SHAP_BACKGROUND_MANIFEST.to_csv(OUTPUT_ROOT / "shap_background_manifest.csv", index=False)
print("IG samples:", len(SAMPLE_MANIFEST), "| sensitivity:", len(SENSITIVITY_MANIFEST),
      "| attention:", len(ATTENTION_MANIFEST), "| SHAP:", len(SHAP_MANIFEST),
      "| SHAP background:", len(SHAP_BACKGROUND_MANIFEST))
del TRAIN_LOADER, TRAIN_DATASET
gc.collect()

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['bio_core', 'spliceai', 'geom']
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.



Window cache dna/train3: 100%|██████████| 11604/11604 [00:06<00:00, 1692.41shard/s]


[Window cache] dna/train3: 23208 samples ready in 6.9s


Window cache protein/train3: 100%|██████████| 1451/1451 [00:01<00:00, 912.68shard/s]
c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict_clean\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[Window cache] protein/train3: 23208 samples ready in 1.6s
Background 2026: 256 train3 examples
Background 2027: 256 train3 examples
Background 2028: 256 train3 examples
Normalized AF values for raw zero [AF, gnomADe_AF]: [-0.60429054 -1.3362284 ]
Train3 normalized AF medians [AF, gnomADe_AF]: [-0.60429054 -0.01186047]
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['bio_core', 'spliceai', 'geom']
[+] Dataset sẵn sàng: 6095 mẫu. Đã tối ưu hóa RAM & CPU 100%.



Window cache dna/test: 100%|██████████| 3048/3048 [00:01<00:00, 1756.02shard/s]


... (14 outputs left out)

[Window cache] protein/proteingym: 1472 samples ready in 0.1s
Train3Val_ProteinGym n= 1472 selected= 185


,True_Label,AF_Stratum,N_Selected
0,0,"(0,1e-5]",25
1,0,"(1e-5,1e-4]",25
2,0,0,25
3,0,>1e-4,25
4,1,"(0,1e-5]",25
5,1,"(1e-5,1e-4]",25
6,1,0,25
7,1,>1e-4,10


IG samples: 535 | sensitivity: 60 | attention: 144 | SHAP: 30 | SHAP background: 32


0

## Run settings and resume

Each stage writes immutable Parquet chunks under `parquet_partitions/<stage>/<dataset>/seed_<seed>/`. A completed chunk is skipped on rerun; CSV exports are consolidated at the end of each stage. The run signature prevents mixing outputs when checkpoints, sample manifests, model settings, or chunk sizes change. For a different run, set a new `OUTPUT_ROOT`.


In [6]:
RUN_SIGNATURE = {
    "source_batch": str(SOURCE_BATCH_DIR.resolve()),
    "source_checkpoints": {
        str(seed): {
            "path": SOURCE_INFO[seed]["checkpoint"],
            "size": Path(SOURCE_INFO[seed]["checkpoint"]).stat().st_size,
            "mtime_ns": Path(SOURCE_INFO[seed]["checkpoint"]).stat().st_mtime_ns,
            "best_epoch": int(SOURCE_INFO[seed]["best_epoch"]),
            "validation_mcc": float(SOURCE_INFO[seed]["val_mcc"]),
            "decision_threshold": float(SOURCE_INFO[seed]["threshold"]),
        }
        for seed in SEEDS_TO_EXPLAIN
    },
    "seeds": list(map(int, SEEDS_TO_EXPLAIN)),
    "datasets": DATASETS_TO_EXPLAIN,
    "model_config": MODEL_CONFIG,
    "bio_feature_names": BIO_FEATURE_NAMES,
    "geometry_feature_names": list(GEOM_FEATURES),
    "sample_manifest_sha256": hashlib.sha256(
        SAMPLE_MANIFEST.to_csv(index=False).encode("utf-8")
    ).hexdigest(),
    "attention_manifest_sha256": hashlib.sha256(
        ATTENTION_MANIFEST.to_csv(index=False).encode("utf-8")
    ).hexdigest(),
    "sensitivity_manifest_sha256": hashlib.sha256(
        SENSITIVITY_MANIFEST.to_csv(index=False).encode("utf-8")
    ).hexdigest(),
    "samples_per_label_af_stratum": SAMPLES_PER_LABEL_AF_STRATUM,
    "attention_samples_per_label": ATTENTION_SAMPLES_PER_LABEL,
    "sensitivity_samples_per_label": SENSITIVITY_SAMPLES_PER_LABEL,
    "background_samples": BACKGROUND_SAMPLES,
    "background_seeds": list(map(int, BACKGROUND_SEEDS)),
    "ig_steps": IG_STEPS,
    "ig_retry_steps": IG_RETRY_STEPS,
    "ig_internal_batch_size": IG_INTERNAL_BATCH_SIZE,
    "ig_chunk_size": IG_CHUNK_SIZE,
    "sensitivity_chunk_size": SENSITIVITY_CHUNK_SIZE,
    "occlusion_chunk_size": OCCLUSION_CHUNK_SIZE,
    "attention_chunk_size": ATTENTION_CHUNK_SIZE,
    "batch_size": BATCH_SIZE,
    "expected_test_mcc": {str(k): v for k, v in EXPECTED_TEST_MCC.items()},
    "mcc_reference_tolerance": MCC_REFERENCE_TOLERANCE,
}
signature_path = OUTPUT_ROOT / "run_signature.json"
if signature_path.is_file():
    previous_signature = json.loads(signature_path.read_text(encoding="utf-8"))
    if previous_signature != RUN_SIGNATURE:
        raise ValueError(
            "Run settings or sample manifests differ from the existing v2 output. "
            "Change OUTPUT_ROOT to a new folder before running; do not mix results."
        )
else:
    signature_path.write_text(json.dumps(RUN_SIGNATURE, indent=2), encoding="utf-8")

STORE = PartitionedParquetStore(OUTPUT_ROOT)
EXPERIMENT = CrossAttentionInterpretabilityExperiment(
    output_root=OUTPUT_ROOT,
    store=STORE,
    seeds=SEEDS_TO_EXPLAIN,
    datasets_to_explain=DATASETS_TO_EXPLAIN,
    sample_manifest=SAMPLE_MANIFEST,
    attention_manifest=ATTENTION_MANIFEST,
    sensitivity_manifest=SENSITIVITY_MANIFEST,
    source_info=SOURCE_INFO,
    model_config=MODEL_CONFIG,
    bio_feature_names=BIO_FEATURE_NAMES,
    geom_feature_names=GEOM_FEATURES,
    device=DEVICE,
    make_loader=make_explanation_loader,
    load_seed_model=load_seed_model,
    background_baselines=BACKGROUND_BASELINES,
    train_feature_medians=TRAIN_FEATURE_MEDIANS,
    af_zero_values=AF_ZERO_VALUES,
    af_median_values=AF_MEDIAN_VALUES,
    ig_steps=IG_STEPS,
    ig_retry_steps=IG_RETRY_STEPS,
    ig_internal_batch_size=IG_INTERNAL_BATCH_SIZE,
    ig_chunk_size=IG_CHUNK_SIZE,
    sensitivity_chunk_size=SENSITIVITY_CHUNK_SIZE,
    occlusion_chunk_size=OCCLUSION_CHUNK_SIZE,
    attention_chunk_size=ATTENTION_CHUNK_SIZE,
    mcc_reference=EXPECTED_TEST_MCC,
    mcc_tolerance=MCC_REFERENCE_TOLERANCE,
    allow_low_background_stability=ALLOW_LOW_BACKGROUND_STABILITY_TO_CONTINUE,
)
print("Resumable v2 experiment ready:", OUTPUT_ROOT)

Resumable v2 experiment ready: d:\Missense_variant_pathogenicity_prediction\experiments\cross_attention_explainability_v2


## A — Dự đoán trên toàn bộ từng tập test

Chạy mỗi checkpoint trên toàn bộ mẫu, lưu logit/xác suất/threshold/dự đoán và tính lại MCC. Trên `Train3Val_Test`, MCC được đối chiếu với số tham chiếu theo seed; sai khác vượt tolerance được in rõ để kiểm tra checkpoint, dữ liệu và threshold.

In [7]:
PREDICTIONS_ALL, PREDICTION_METRICS = EXPERIMENT.run_predictions()
display(PREDICTION_METRICS)
failed_reference = PREDICTION_METRICS[
    PREDICTION_METRICS["MCC_Match_Within_Tolerance"].eq(False)
]
if not failed_reference.empty:
    print("[CHECK] At least one recomputed Test MCC differs from its reference beyond tolerance.")

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['bio_core', 'spliceai', 'geom']
[+] Dataset sẵn sàng: 6095 mẫu. Đã tối ưu hóa RAM & CPU 100%.



Window cache dna/test: 100%|██████████| 3048/3048 [00:01<00:00, 1633.84shard/s]


[Window cache] dna/test: 6095 samples ready in 1.9s


Window cache protein/test: 100%|██████████| 381/381 [00:00<00:00, 941.13shard/s]


[Window cache] protein/test: 6095 samples ready in 0.4s
[RESUME] predictions Train3Val_Test seed=5: complete
[RESUME] predictions Train3Val_Test seed=6: complete
[RESUME] predictions Train3Val_Test seed=7: complete
[RESUME] predictions Train3Val_Test seed=37: complete
[RESUME] predictions Train3Val_Test seed=42: complete
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['bio_core', 'spliceai', 'geom']
[+] Dataset sẵn sàng: 703 mẫu. Đã tối ưu hóa RAM & CPU 100%.



Window cache dna/clinvarhq: 100%|██████████| 352/352 [00:00<00:00, 1787.83shard/s]


... (5 outputs left out)

Window cache protein/proteingym: 100%|██████████| 92/92 [00:00<00:00, 962.08shard/s]


[Window cache] protein/proteingym: 1472 samples ready in 0.1s
[RESUME] predictions Train3Val_ProteinGym seed=5: complete
[RESUME] predictions Train3Val_ProteinGym seed=6: complete
[RESUME] predictions Train3Val_ProteinGym seed=7: complete
[RESUME] predictions Train3Val_ProteinGym seed=37: complete
[RESUME] predictions Train3Val_ProteinGym seed=42: complete
[A] Recomputed MCC and reference check:


,Seed,Dataset,N,MCC,Reference_MCC,Absolute_Difference,MCC_Match_Within_Tolerance,Decision_Threshold
0,5,Train3Val_ClinVarHQ,703,0.878819,NaN,NaN,NaN,0.888212
1,5,Train3Val_ProteinGym,1472,0.696304,NaN,NaN,NaN,0.888212
2,5,Train3Val_Test,6095,0.838118,0.8381,0.000018,True,0.888212
3,6,Train3Val_ClinVarHQ,703,0.866350,NaN,NaN,NaN,0.918221
4,6,Train3Val_ProteinGym,1472,0.671839,NaN,NaN,NaN,0.918221
5,6,Train3Val_Test,6095,0.831597,0.8316,0.000003,True,0.918221
6,7,Train3Val_ClinVarHQ,703,0.879155,NaN,NaN,NaN,0.559536
7,7,Train3Val_ProteinGym,1472,0.694199,NaN,NaN,NaN,0.559536
8,7,Train3Val_Test,6095,0.839465,0.8395,0.000035,True,0.559536
9,37,Train3Val_ClinVarHQ,703,0.894255,NaN,NaN,NaN,0.479132


,Seed,Dataset,N,MCC,Reference_MCC,Absolute_Difference,MCC_Match_Within_Tolerance,Decision_Threshold
0,5,Train3Val_ClinVarHQ,703,0.878819,NaN,NaN,NaN,0.888212
1,5,Train3Val_ProteinGym,1472,0.696304,NaN,NaN,NaN,0.888212
2,5,Train3Val_Test,6095,0.838118,0.8381,0.000018,True,0.888212
3,6,Train3Val_ClinVarHQ,703,0.866350,NaN,NaN,NaN,0.918221
4,6,Train3Val_ProteinGym,1472,0.671839,NaN,NaN,NaN,0.918221
5,6,Train3Val_Test,6095,0.831597,0.8316,0.000003,True,0.918221
6,7,Train3Val_ClinVarHQ,703,0.879155,NaN,NaN,NaN,0.559536
7,7,Train3Val_ProteinGym,1472,0.694199,NaN,NaN,NaN,0.559536
8,7,Train3Val_Test,6095,0.839465,0.8395,0.000035,True,0.559536
9,37,Train3Val_ClinVarHQ,703,0.894255,NaN,NaN,NaN,0.479132


## D — Can thiệp AF trên toàn bộ mẫu

So sánh đầu vào gốc với hai can thiệp: đặt AF và gnomADe_AF thô bằng 0 rồi áp dụng đúng log-transform/scaler đã fit; hoặc thay bằng median chuẩn hóa của Train3. Mọi feature khác được giữ nguyên và threshold theo seed không đổi.

In [8]:
AF_INTERVENTIONS, AF_INTERVENTION_METRICS = EXPERIMENT.run_af_interventions()
display(AF_INTERVENTION_METRICS)

[RESUME] AF interventions Train3Val_Test seed=5: complete
[RESUME] AF interventions Train3Val_Test seed=6: complete
[RESUME] AF interventions Train3Val_Test seed=7: complete
[RESUME] AF interventions Train3Val_Test seed=37: complete
[RESUME] AF interventions Train3Val_Test seed=42: complete
[RESUME] AF interventions Train3Val_ClinVarHQ seed=5: complete
[RESUME] AF interventions Train3Val_ClinVarHQ seed=6: complete
[RESUME] AF interventions Train3Val_ClinVarHQ seed=7: complete
[RESUME] AF interventions Train3Val_ClinVarHQ seed=37: complete
[RESUME] AF interventions Train3Val_ClinVarHQ seed=42: complete
[RESUME] AF interventions Train3Val_ProteinGym seed=5: complete
[RESUME] AF interventions Train3Val_ProteinGym seed=6: complete
[RESUME] AF interventions Train3Val_ProteinGym seed=7: complete
[RESUME] AF interventions Train3Val_ProteinGym seed=37: complete
[RESUME] AF interventions Train3Val_ProteinGym seed=42: complete


,Seed,Dataset,Scenario,N,MCC,Accuracy,AUROC,AUPRC,Mean_Absolute_Logit_Change_From_Original,Prediction_Flip_Rate_From_Original,Decision_Threshold
0,5,Train3Val_ClinVarHQ,af_median,703,0.702437,0.850640,0.918140,0.889440,2.417916,0.140825,0.888212
1,5,Train3Val_ClinVarHQ,af_zero,703,0.664437,0.815078,0.937559,0.920216,3.160911,0.204836,0.888212
2,5,Train3Val_ClinVarHQ,original,703,0.878819,0.937411,0.989914,0.990126,0.000000,0.000000,0.888212
3,5,Train3Val_ProteinGym,af_median,1472,0.651116,0.826766,0.899440,0.854429,1.754610,0.124321,0.888212
4,5,Train3Val_ProteinGym,af_zero,1472,0.642064,0.805707,0.920799,0.895127,1.248512,0.091033,0.888212
5,5,Train3Val_ProteinGym,original,1472,0.696304,0.846467,0.926479,0.919061,0.000000,0.000000,0.888212
6,5,Train3Val_Test,af_median,6095,0.701415,0.880394,0.935761,0.842039,1.814287,0.087449,0.888212
7,5,Train3Val_Test,af_zero,6095,0.671638,0.842658,0.946006,0.877290,2.609638,0.128138,0.888212
8,5,Train3Val_Test,original,6095,0.838118,0.936669,0.973359,0.946605,0.000000,0.000000,0.888212
9,6,Train3Val_ClinVarHQ,af_median,703,0.602419,0.800853,0.893589,0.855278,2.595918,0.186344,0.918221


## B — IG theo AF strata

Chọn tối đa 25 mẫu cho mỗi tổ hợp dataset × nhãn × AF stratum từ AF thô: `0`, `(0, 1e-5]`, `(1e-5, 1e-4]`, `>1e-4`. Cùng một manifest được dùng cho cả 5 seed; `Sampling_Weight` cho phép ước lượng lại trung bình theo phân bố nguồn. Mỗi mẫu có hai phép IG: dự đoán đầy đủ với nền train3, và `alt_vs_ref` giữ các feature khác cố định. Mặc định 64 bước; tự tính lại bằng 128 bước nếu residual tương đối vượt 5%.

In [9]:
IG_RESULTS = EXPERIMENT.run_ig()
for stage_name, result_frame in IG_RESULTS.items():
    print(stage_name, result_frame.shape)

[B] residuals still >5% after adaptive 64/128 steps: 35/5350
ig_modalities (21400, 19)
ig_positions (706200, 21)
ig_features (149800, 20)
ig_completeness (5350, 19)


## B — Kiểm tra độ nhạy với background IG

Trên 20 mẫu mỗi dataset (10 benign + 10 pathogenic, rải theo AF strata), so sánh thứ hạng attribution của background 2026 với hai background độc lập 2027 và 2028. Cell lưu Spearman theo dataset × seed × background comparison. C và E yêu cầu mọi so sánh đạt 0.8; nếu chưa đạt, xem `ig_background_stability.csv` trước khi bật override.

In [10]:
BACKGROUND_STABILITY = EXPERIMENT.run_ig_background_sensitivity()
display(BACKGROUND_STABILITY)

[B-sensitivity] Feature/token importance rank stability:


,Dataset,Seed,Primary_Background_Seed,Comparison_Background_Seed,Spearman_Rank_Correlation,N_Units,Pass_0_8
0,Train3Val_Test,5,2026,2027,0.998345,160,True
1,Train3Val_Test,5,2026,2028,0.998099,160,True
2,Train3Val_Test,6,2026,2027,0.996042,160,True
3,Train3Val_Test,6,2026,2028,0.999285,160,True
4,Train3Val_Test,7,2026,2027,0.998986,160,True
5,Train3Val_Test,7,2026,2028,0.998585,160,True
6,Train3Val_Test,37,2026,2027,0.998942,160,True
7,Train3Val_Test,37,2026,2028,0.999238,160,True
8,Train3Val_Test,42,2026,2027,0.998005,160,True
9,Train3Val_Test,42,2026,2028,0.997996,160,True


[GATE] All background comparisons have Spearman >=0.8.


,Dataset,Seed,Primary_Background_Seed,Comparison_Background_Seed,Spearman_Rank_Correlation,N_Units,Pass_0_8
0,Train3Val_Test,5,2026,2027,0.998345,160,True
1,Train3Val_Test,5,2026,2028,0.998099,160,True
2,Train3Val_Test,6,2026,2027,0.996042,160,True
3,Train3Val_Test,6,2026,2028,0.999285,160,True
4,Train3Val_Test,7,2026,2027,0.998986,160,True
5,Train3Val_Test,7,2026,2028,0.998585,160,True
6,Train3Val_Test,37,2026,2027,0.998942,160,True
7,Train3Val_Test,37,2026,2028,0.999238,160,True
8,Train3Val_Test,42,2026,2027,0.998005,160,True
9,Train3Val_Test,42,2026,2028,0.997996,160,True


## C — Occlusion cá thể và theo nhóm

Trên cùng manifest IG, đo thay đổi logit khi thay từng feature/tabular value bằng median train3, đổi từng token alt thành token ref, và can thiệp các nhóm AF, conservation, SpliceAI, geometry, protein center, protein neighbors, DNA window. Kết quả được tách thành file cá thể và nhóm.

In [11]:
OCCLUSION_INDIVIDUAL, OCCLUSION_GROUPS = EXPERIMENT.run_occlusion()
print("Individual occlusion:", OCCLUSION_INDIVIDUAL.shape)
print("Grouped occlusion:", OCCLUSION_GROUPS.shape)

Individual occlusion: (251450, 16)
Grouped occlusion: (18725, 16)


## E — Attention maps

Trên mỗi dataset lấy 24 benign + 24 pathogenic, rải theo AF strata; danh sách mẫu cân bằng này dùng chung cho cả 5 seed. Cell lưu toàn bộ map layer/head/query/key dạng float16 `.npz`, đồng thời tạo bảng tóm tắt query `Protein:+0` và `gnomADe_AF` theo context token.

In [12]:
ATTENTION_SUMMARY = EXPERIMENT.run_attention()
print("Attention summary:", ATTENTION_SUMMARY.shape)

Attention summary: (274320, 16)


## SHAP — Exact, Gradient và Deep

SHAP chạy trên một mẫu phụ có 5 benign + 5 pathogenic mỗi dataset (có thể chỉnh ở cell cấu hình) và 32 mẫu background train3. `ExactExplainer` tính chính xác Shapley trên **4 nhóm** DNA, protein, bio/SpliceAI và geometry (16 coalitions), với baseline mean-token/median-feature; không chạy exact trên hàng nghìn chiều token vì số coalitions tăng theo hàm mũ. `GradientExplainer` dùng expected gradients và background train3; `DeepExplainer` dùng Deep SHAP trên cùng background. Hai phương pháp gradient/deep giữ attribution riêng cho modality, allele/vị trí token và từng feature.

Đích giải thích là pathogenic logit trước sigmoid. Kết quả SHAP là xấp xỉ theo background sampling đối với Gradient/Deep; Mask, variant index và pooled metadata được giữ cố định theo mẫu đang giải thích. DeepExplainer có thể không hỗ trợ một phép toán nội bộ của PyTorch trong model này. Cell ghi trạng thái/lỗi theo từng mẫu và tiếp tục các explainer còn lại. Nếu Deep additivity check thất bại, code thử lại với check tắt nhưng lưu residual riêng để không che sai khác. Raw tensor attribution lưu `.npz`; bảng gọn lưu CSV/Parquet để resume.

In [ ]:
# Allow running this SHAP cell after the existing A-E cells without repeating them.
import shap
SHAP_VERSION = str(getattr(shap, "__version__", "unknown"))
if "SAMPLE_MANIFEST" not in globals():
    SAMPLE_MANIFEST = pd.read_csv(OUTPUT_ROOT / "sample_manifest.csv")
SHAP_MANIFEST = select_spread_subset(
    SAMPLE_MANIFEST, per_label=SHAP_SAMPLES_PER_LABEL, seed=SHAP_BACKGROUND_SEED
)
expected_background_signature = {
    "samples": SHAP_BACKGROUND_SAMPLES, "seed": SHAP_BACKGROUND_SEED,
}
if (
    "SHAP_BACKGROUND_INPUTS" not in globals()
    or globals().get("SHAP_BACKGROUND_SIGNATURE") != expected_background_signature
):
    shap_train_loader = make_explanation_loader("train3", SEEDS_TO_EXPLAIN[0])
    shap_train_dataset = shap_train_loader.dataset
    if "BACKGROUND_BASELINES" not in globals() or 2026 not in BACKGROUND_BASELINES:
        primary_background_indices = select_stratified_indices(
            shap_train_dataset, BACKGROUND_SAMPLES, seed=2026
        )
        primary_background_batch = shap_train_dataset[primary_background_indices]
        if "BACKGROUND_BASELINES" not in globals():
            BACKGROUND_BASELINES = {}
        BACKGROUND_BASELINES[2026] = empirical_baseline(
            primary_background_batch, primary_background_batch
        )
        del primary_background_batch
    shap_background_indices = select_stratified_indices(
        shap_train_dataset, SHAP_BACKGROUND_SAMPLES, seed=SHAP_BACKGROUND_SEED
    )
    shap_background_batch = shap_train_dataset[shap_background_indices]
    SHAP_BACKGROUND_INPUTS = {
        key: shap_background_batch[key].to(torch.float32).cpu()
        for key in ("token_dna", "token_protein", "bio_features", "geom_features")
    }
    train_label_column = next(
        name for name in ("Pathogenicity_Label", "Label", "label", "target", "Target")
        if name in shap_train_dataset.df.columns
    )
    SHAP_BACKGROUND_MANIFEST = shap_train_dataset.df.iloc[shap_background_indices][
        ["Variant_ID", train_label_column]
    ].reset_index(drop=True).rename(columns={train_label_column: "True_Label"})
    SHAP_BACKGROUND_MANIFEST.insert(0, "Sample_Index", shap_background_indices)
    SHAP_BACKGROUND_MANIFEST["Background_Seed"] = SHAP_BACKGROUND_SEED
    SHAP_BACKGROUND_SIGNATURE = expected_background_signature
    del shap_background_batch, shap_train_dataset, shap_train_loader
    gc.collect()

existing_shap_signature_path = OUTPUT_ROOT / "shap_run_signature.json"
candidate_sample_hash = hashlib.sha256(
    SHAP_MANIFEST.to_csv(index=False).encode("utf-8")
).hexdigest()
candidate_background_hash = hashlib.sha256(
    SHAP_BACKGROUND_MANIFEST.to_csv(index=False).encode("utf-8")
).hexdigest()
if existing_shap_signature_path.is_file():
    existing_shap_signature = json.loads(
        existing_shap_signature_path.read_text(encoding="utf-8")
    )
    if (
        existing_shap_signature.get("sample_manifest_sha256") != candidate_sample_hash
        or existing_shap_signature.get("background_manifest_sha256") != candidate_background_hash
    ):
        raise ValueError("SHAP sample/background manifest changed; use a new OUTPUT_ROOT.")
SHAP_MANIFEST.to_csv(OUTPUT_ROOT / "shap_sample_manifest.csv", index=False)
SHAP_BACKGROUND_MANIFEST.to_csv(OUTPUT_ROOT / "shap_background_manifest.csv", index=False)

SHAP_SIGNATURE = {
    "base_run_signature_sha256": hashlib.sha256(
        (OUTPUT_ROOT / "run_signature.json").read_bytes()
    ).hexdigest(),
    "explainer_names": ["ExactExplainer", "GradientExplainer", "DeepExplainer"],
    "shap_version": SHAP_VERSION,
    "samples_per_label_per_dataset": SHAP_SAMPLES_PER_LABEL,
    "sample_manifest_sha256": hashlib.sha256(
        SHAP_MANIFEST.to_csv(index=False).encode("utf-8")
    ).hexdigest(),
    "background_samples": SHAP_BACKGROUND_SAMPLES,
    "background_seed": SHAP_BACKGROUND_SEED,
    "background_manifest_sha256": hashlib.sha256(
        SHAP_BACKGROUND_MANIFEST.to_csv(index=False).encode("utf-8")
    ).hexdigest(),
    "gradient_nsamples": SHAP_GRADIENT_NSAMPLES,
    "gradient_batch_size": SHAP_GRADIENT_BATCH_SIZE,
    "chunk_size": SHAP_CHUNK_SIZE,
    "deep_additivity_fallback": SHAP_DEEP_ADDITIVITY_FALLBACK,
    "exact_groups": ["DNA", "Protein", "Bio_SpliceAI", "Geometry"],
    "exact_baseline": "train3_background_mean_tokens_and_median_features_seed_2026",
    "output": "pathogenic_logit_before_sigmoid",
}
shap_signature_path = OUTPUT_ROOT / "shap_run_signature.json"
if shap_signature_path.is_file():
    previous = json.loads(shap_signature_path.read_text(encoding="utf-8"))
    if previous != SHAP_SIGNATURE:
        raise ValueError(
            "SHAP settings or manifests differ from the existing SHAP results. "
            "Change OUTPUT_ROOT before running SHAP again."
        )
else:
    shap_signature_path.write_text(json.dumps(SHAP_SIGNATURE, indent=2), encoding="utf-8")

SHAP_EXPERIMENT = CrossAttentionShapExperiment(
    output_root=OUTPUT_ROOT,
    store=STORE,
    seeds=SEEDS_TO_EXPLAIN,
    datasets_to_explain=DATASETS_TO_EXPLAIN,
    sample_manifest=SHAP_MANIFEST,
    source_info=SOURCE_INFO,
    model_config=MODEL_CONFIG,
    bio_feature_names=BIO_FEATURE_NAMES,
    geom_feature_names=GEOM_FEATURES,
    device=DEVICE,
    make_loader=make_explanation_loader,
    load_seed_model=load_seed_model,
    background_inputs=SHAP_BACKGROUND_INPUTS,
    exact_baseline_inputs=BACKGROUND_BASELINES[2026],
    gradient_nsamples=SHAP_GRADIENT_NSAMPLES,
    gradient_batch_size=SHAP_GRADIENT_BATCH_SIZE,
    chunk_size=SHAP_CHUNK_SIZE,
    deep_additivity_fallback=SHAP_DEEP_ADDITIVITY_FALLBACK,
)
SHAP_RESULTS = SHAP_EXPERIMENT.run()
for result_name, frame in SHAP_RESULTS.items():
    print(result_name, frame.shape)
SHAP_STATUS = pd.concat(
    [SHAP_RESULTS[name] for name in (
        "shap_exact_status", "shap_gradient_status", "shap_deep_status"
    ) if name in SHAP_RESULTS],
    ignore_index=True,
)
display(SHAP_STATUS.groupby(["Explainer", "Status"], dropna=False).size().rename("N").reset_index())
display(SHAP_STATUS[SHAP_STATUS["Status"].eq("error")][
    ["Explainer", "Dataset", "Seed", "Variant_ID", "Error_Type", "Error_Message"]
].head(20))

## Tổng hợp

Các phép IG, occlusion, attention và SHAP được đọc từ các chunk Parquet đã lưu, vì vậy cell này có thể chạy lại độc lập sau khi các giai đoạn hoàn tất. Tóm tắt IG/occlusion giữ từng seed và dùng `Sampling_Weight` để điều chỉnh việc lấy mẫu theo AF; không gộp các seed thành một model. Attention và SHAP được tổng hợp trên các tập mẫu phụ đã chọn, không xem là ước lượng toàn bộ quần thể. Exact SHAP chỉ phân rã bốn nhóm modality; Gradient/Deep có thêm bảng theo token và feature.


In [ ]:
def _read_output(filename):
    path = OUTPUT_ROOT / filename
    return pd.read_csv(path) if path.is_file() else pd.DataFrame()


PREDICTION_METRICS = _read_output("prediction_metrics_by_seed.csv")
AF_INTERVENTION_METRICS = _read_output("af_intervention_metrics.csv")
IG_MODALITIES = _read_output("ig_modality_attributions.csv")
IG_POSITIONS = _read_output("ig_token_positions.csv")
IG_FEATURES = _read_output("ig_tabular_features.csv")
IG_COMPLETENESS = _read_output("ig_completeness.csv")
IG_BACKGROUND_STABILITY = _read_output("ig_background_stability.csv")
OCCLUSION_INDIVIDUAL = _read_output("occlusion_individual.csv")
OCCLUSION_GROUPS = _read_output("occlusion_groups.csv")
ATTENTION_SUMMARY = _read_output("attention_summary.csv")
SHAP_EXACT_GROUPS = _read_output("shap_exact_groups.csv")
SHAP_EXACT_STATUS = _read_output("shap_exact_status.csv")
SHAP_GRADIENT_MODALITIES = _read_output("shap_gradient_modalities.csv")
SHAP_GRADIENT_POSITIONS = _read_output("shap_gradient_positions.csv")
SHAP_GRADIENT_FEATURES = _read_output("shap_gradient_features.csv")
SHAP_GRADIENT_COMPLETENESS = _read_output("shap_gradient_completeness.csv")
SHAP_GRADIENT_STATUS = _read_output("shap_gradient_status.csv")
SHAP_DEEP_MODALITIES = _read_output("shap_deep_modalities.csv")
SHAP_DEEP_POSITIONS = _read_output("shap_deep_positions.csv")
SHAP_DEEP_FEATURES = _read_output("shap_deep_features.csv")
SHAP_DEEP_COMPLETENESS = _read_output("shap_deep_completeness.csv")
SHAP_DEEP_STATUS = _read_output("shap_deep_status.csv")


def _weighted_summary(frame, group_columns, value_columns):
    if frame.empty:
        return pd.DataFrame()
    rows = []
    grouper = group_columns[0] if len(group_columns) == 1 else group_columns
    for key, group in frame.groupby(grouper, dropna=False, sort=False):
        key = (key,) if len(group_columns) == 1 else tuple(key)
        weights = pd.to_numeric(group.get("Sampling_Weight", 1.0), errors="coerce")
        if not isinstance(weights, pd.Series):
            weights = pd.Series(float(weights), index=group.index)
        weights = weights.fillna(1.0).astype(float).to_numpy()
        row = dict(zip(group_columns, key))
        row["N_Samples"] = int(group["Variant_ID"].nunique()) if "Variant_ID" in group else len(group)
        for column in value_columns:
            values = pd.to_numeric(group[column], errors="coerce").to_numpy(dtype=float)
            valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
            if not valid.any():
                row[f"Weighted_Mean_{column}"] = np.nan
                row[f"Weighted_SD_{column}"] = np.nan
                continue
            mean = float(np.average(values[valid], weights=weights[valid]))
            variance = float(np.average((values[valid] - mean) ** 2, weights=weights[valid]))
            row[f"Weighted_Mean_{column}"] = mean
            row[f"Weighted_SD_{column}"] = float(np.sqrt(variance))
        rows.append(row)
    return pd.DataFrame(rows)


if not IG_MODALITIES.empty:
    modality_summary = _weighted_summary(
        IG_MODALITIES,
        ["Dataset", "Seed", "Attribution_Mode", "True_Label", "Modality"],
        ["Signed_Attribution_Sum", "Absolute_Attribution_Sum"],
    )
    modality_summary.to_csv(OUTPUT_ROOT / "ig_modality_summary_weighted.csv", index=False)
    display(modality_summary)

if not IG_POSITIONS.empty:
    position_summary = _weighted_summary(
        IG_POSITIONS,
        ["Dataset", "Seed", "Attribution_Mode", "True_Label", "Modality", "Allele",
         "Relative_Token_Position"],
        ["Signed_Attribution", "Absolute_Attribution"],
    )
    position_summary.to_csv(OUTPUT_ROOT / "ig_token_position_summary_weighted.csv", index=False)
    display(position_summary.head(40))

if not IG_FEATURES.empty:
    feature_summary = _weighted_summary(
        IG_FEATURES,
        ["Dataset", "Seed", "Attribution_Mode", "True_Label", "Feature"],
        ["Signed_Attribution", "Absolute_Attribution"],
    )
    feature_summary.to_csv(OUTPUT_ROOT / "ig_feature_summary_weighted.csv", index=False)
    for key, group in feature_summary.groupby(
        ["Dataset", "Seed", "Attribution_Mode", "True_Label"], sort=False
    ):
        print("Top features", key)
        display(group.nlargest(10, "Weighted_Mean_Absolute_Attribution"))

if not IG_COMPLETENESS.empty:
    print("IG completeness residuals by baseline / step count:")
    display(IG_COMPLETENESS.groupby(
        ["Dataset", "Seed", "Attribution_Mode", "IG_Steps_Used", "IG_Residual_Over_5pct"],
        dropna=False,
    ).size().rename("N").reset_index())

if not IG_BACKGROUND_STABILITY.empty:
    display(IG_BACKGROUND_STABILITY)

if not OCCLUSION_INDIVIDUAL.empty:
    individual_summary = _weighted_summary(
        OCCLUSION_INDIVIDUAL,
        ["Dataset", "Seed", "True_Label", "Feature_or_Position"],
        ["Logit_Delta_Actual_Minus_Occluded"],
    )
    individual_summary["Weighted_Mean_Absolute_Logit_Delta"] = _weighted_summary(
        OCCLUSION_INDIVIDUAL.assign(
            Absolute_Logit_Delta=OCCLUSION_INDIVIDUAL[
                "Logit_Delta_Actual_Minus_Occluded"
            ].abs()
        ),
        ["Dataset", "Seed", "True_Label", "Feature_or_Position"],
        ["Absolute_Logit_Delta"],
    )["Weighted_Mean_Absolute_Logit_Delta"]
    individual_summary.to_csv(OUTPUT_ROOT / "occlusion_individual_summary_weighted.csv", index=False)
    display(individual_summary.head(40))

if not OCCLUSION_GROUPS.empty:
    group_summary = _weighted_summary(
        OCCLUSION_GROUPS,
        ["Dataset", "Seed", "True_Label", "Feature_or_Position"],
        ["Logit_Delta_Actual_Minus_Occluded"],
    )
    grouped_absolute = _weighted_summary(
        OCCLUSION_GROUPS.assign(
            Absolute_Logit_Delta=OCCLUSION_GROUPS[
                "Logit_Delta_Actual_Minus_Occluded"
            ].abs()
        ),
        ["Dataset", "Seed", "True_Label", "Feature_or_Position"],
        ["Absolute_Logit_Delta"],
    )
    group_summary["Weighted_Mean_Absolute_Logit_Delta"] = grouped_absolute[
        "Weighted_Mean_Absolute_Logit_Delta"
    ]
    group_summary.to_csv(OUTPUT_ROOT / "occlusion_group_summary_weighted.csv", index=False)
    display(group_summary)

if not ATTENTION_SUMMARY.empty:
    attention_summary = (
        ATTENTION_SUMMARY.groupby(
            ["Dataset", "Seed", "True_Label", "AF_Stratum", "Query_Token",
             "Query_Modality", "Context_Modality", "Context_Token"],
            dropna=False,
        )["Mean_Attention_Over_Heads"].mean().rename("Mean_Attention_Over_Samples").reset_index()
    )
    attention_summary.to_csv(OUTPUT_ROOT / "attention_summary_by_label_af.csv", index=False)
    display(attention_summary.head(40))

print("Prediction metrics:")
display(PREDICTION_METRICS)
print("AF intervention metrics:")
display(AF_INTERVENTION_METRICS)
print("Outputs:", OUTPUT_ROOT)
for explainer_name, status_frame in (
    ("ExactExplainer", SHAP_EXACT_STATUS),
    ("GradientExplainer", SHAP_GRADIENT_STATUS),
    ("DeepExplainer", SHAP_DEEP_STATUS),
):
    if not status_frame.empty:
        print(explainer_name, "status")
        display(status_frame.groupby("Status", dropna=False).size().rename("N").reset_index())

if not SHAP_EXACT_GROUPS.empty:
    display(SHAP_EXACT_GROUPS.head(40))

for explainer_name, modality_frame, position_frame, feature_frame, completeness_frame in (
    ("GradientExplainer", SHAP_GRADIENT_MODALITIES, SHAP_GRADIENT_POSITIONS,
     SHAP_GRADIENT_FEATURES, SHAP_GRADIENT_COMPLETENESS),
    ("DeepExplainer", SHAP_DEEP_MODALITIES, SHAP_DEEP_POSITIONS,
     SHAP_DEEP_FEATURES, SHAP_DEEP_COMPLETENESS),
):
    if not modality_frame.empty:
        modality_summary = modality_frame.groupby(
            ["Dataset", "Seed", "True_Label", "Modality"], dropna=False
        )["Absolute_Attribution_Sum"].mean().rename("Mean_Absolute_SHAP_Sum").reset_index()
        modality_summary["Explainer"] = explainer_name
        modality_summary.to_csv(
            OUTPUT_ROOT / f"{explainer_name.lower()}_modality_summary.csv", index=False
        )
        display(modality_summary)
    if not feature_frame.empty:
        feature_summary_shap = feature_frame.groupby(
            ["Dataset", "Seed", "True_Label", "Feature"], dropna=False
        )["Absolute_Attribution"].mean().rename("Mean_Absolute_SHAP").reset_index()
        feature_summary_shap.to_csv(
            OUTPUT_ROOT / f"{explainer_name.lower()}_feature_summary.csv", index=False
        )
        display(feature_summary_shap.sort_values(
            "Mean_Absolute_SHAP", ascending=False
        ).groupby(["Dataset", "Seed", "True_Label"], sort=False).head(10))
    if not completeness_frame.empty:
        print(explainer_name, "SHAP additivity residuals")
        display(completeness_frame.groupby(
            ["Dataset", "Seed"], dropna=False
        )["Relative_Additivity_Residual"].agg(["mean", "median", "max", "count"]).reset_index())


## C?ch ??c v? ??i chi?u c?c ph??ng ph?p gi?i th?ch

Cell h?nh ?nh cu?i t?o th?m: (1) SHAP beeswarm cho 28 biological/geometry features, trong ?? tr?c ngang l? t?c ??ng c? d?u l?n pathogenic logit v? m?u l? gi? tr? feature; (2) waterfall cho t?ng m?u ??i di?n, ph?n r? t? baseline logit ??n logit d? ?o?n; (3) heatmap attribution token c? d?u theo t?ng m?u, modality, allele v? v? tr?; (4) mean ? SD importance qua c?c seed; (5) b?ng/scatter t??ng quan IG, occlusion v? Gradient/Deep SHAP tr?n c?ng variant, seed, feature ho?c token; (6) heatmap lu?ng attention gi?a c?c modality. Heatmap attention theo token d?ng chung thang PowerNorm v? c?t thang m?u ? percentile 99.5; c?c tr?ng s? c?c ??i v?n c? trong CSV nh?ng kh?ng l?m ph?n c?n l?i t?i ?en.

D?u d??ng ngh?a l? t?ng pathogenic logit, d?u ?m ngh?a l? gi?m logit. Occlusion ?o ch?nh l?ch gi?a input quan s?t v? input b? thay b?ng gi? tr? tham chi?u. Attention weight ch? cho bi?t query ??c context n?o, kh?ng ph?i b?ng ch?ng r?ng context ?? l?m d? ?o?n t?ng/gi?m. B?ng ??i chi?u l? ki?m tra m?c ??ng thu?n v? th? h?ng/h??ng; ?? l?n attribution kh?ng th? so tr?c ti?p tuy?t ??i v? baseline v? ph?p can thi?p kh?c nhau. Beeswarm/SHAP seed chart ph?n ?nh t?p SHAP m?u ph? hi?n c?, kh?ng ph?i ??c l??ng to?n b? test population.

C?c h?nh v? CSV b? sung ???c l?u trong `OUTPUT_ROOT/interpretability_comparison/`. N?u Gradient/Deep SHAP tr??c ?? b?o l?i, ch?y l?i cell SHAP tr??c cell t?ng h?p v? cell h?nh cu?i; Exact SHAP ?? ho?n t?t s? ???c resume/skip.

In [ ]:
# Change these selectors to inspect a different dataset, checkpoint, or label.
PLOT_DATASET = "Train3Val_Test"
PLOT_SEED = 42
PLOT_MODE = "full_prediction"
PLOT_LABEL = None  # None shows both classes; otherwise use 0 (benign) or 1 (pathogenic).
PLOT_ATTENTION_LAYER = 3
PLOT_TOP_FEATURES = 15

def _filter_plot(frame):
    if frame.empty:
        return frame
    result = frame[
        frame["Dataset"].astype(str).eq(PLOT_DATASET)
        & pd.to_numeric(frame["Seed"], errors="coerce").eq(int(PLOT_SEED))
    ].copy()
    if PLOT_LABEL is not None and "True_Label" in result:
        result = result[pd.to_numeric(result["True_Label"], errors="coerce").eq(int(PLOT_LABEL))]
    return result


plot_positions = _filter_plot(position_summary if "position_summary" in globals() else pd.DataFrame())
if not plot_positions.empty:
    plot_positions = plot_positions[plot_positions["Attribution_Mode"].eq(PLOT_MODE)]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
    for ax, modality in zip(axes, ("DNA", "Protein")):
        subset = plot_positions[
            plot_positions["Modality"].eq(modality) & plot_positions["Allele"].eq("Alt")
        ]
        for label, group in subset.groupby("True_Label", sort=True):
            curve = group.groupby("Relative_Token_Position")[
                "Weighted_Mean_Absolute_Attribution"
            ].mean()
            ax.plot(curve.index, curve.values, marker="o", markersize=3,
                    label="Benign" if int(label) == 0 else "Pathogenic")
        ax.axvline(0, color="black", linestyle="--", linewidth=1)
        ax.set_title(f"{modality} alt-token IG")
        ax.set_xlabel("Relative token position (variant at 0)")
        ax.grid(alpha=0.25)
        ax.legend()
    axes[0].set_ylabel("AF-weighted mean |IG attribution|")
    fig.suptitle(f"IG by token position | {PLOT_DATASET} | seed {PLOT_SEED} | {PLOT_MODE}")
    fig.tight_layout()
    fig.savefig(OUTPUT_ROOT / f"ig_positions_{PLOT_DATASET}_seed{PLOT_SEED}.png", dpi=160)
    plt.show()

plot_features = _filter_plot(feature_summary if "feature_summary" in globals() else pd.DataFrame())
if not plot_features.empty:
    plot_features = plot_features[plot_features["Attribution_Mode"].eq(PLOT_MODE)]
    plot_features = plot_features.groupby("Feature")[
        "Weighted_Mean_Absolute_Attribution"
    ].mean().nlargest(PLOT_TOP_FEATURES).sort_values()
    if not plot_features.empty:
        fig, ax = plt.subplots(figsize=(9, 6))
        plot_features.plot.barh(ax=ax)
        ax.set_xlabel("AF-weighted mean |IG attribution|")
        ax.set_title(f"Top tabular features | {PLOT_DATASET} | seed {PLOT_SEED} | {PLOT_MODE}")
        fig.tight_layout()
        fig.savefig(OUTPUT_ROOT / f"ig_features_{PLOT_DATASET}_seed{PLOT_SEED}.png", dpi=160)
        plt.show()

plot_groups = _filter_plot(group_summary if "group_summary" in globals() else pd.DataFrame())
if not plot_groups.empty:
    plot_groups = plot_groups.groupby("Feature_or_Position")[
        "Weighted_Mean_Absolute_Logit_Delta"
    ].mean().sort_values()
    fig, ax = plt.subplots(figsize=(9, 5))
    plot_groups.plot.barh(ax=ax)
    ax.set_xlabel("AF-weighted mean |logit change|")
    ax.set_title(f"Grouped occlusion | {PLOT_DATASET} | seed {PLOT_SEED}")
    fig.tight_layout()
    fig.savefig(OUTPUT_ROOT / f"occlusion_groups_{PLOT_DATASET}_seed{PLOT_SEED}.png", dpi=160)
    plt.show()

attention_plot_rows = _filter_plot(ATTENTION_SUMMARY)
if not attention_plot_rows.empty:
    attention_plot_rows = attention_plot_rows[
        attention_plot_rows["Query_Token"].isin(["Protein:+0", "gnomADe_AF"])
    ]
    for query_token, query_rows in attention_plot_rows.groupby("Query_Token", sort=False):
        means = query_rows.groupby(
            ["True_Label", "Context_Modality", "Context_Token"], dropna=False
        )["Mean_Attention_Over_Heads"].mean().reset_index()
        for context_modality, context_rows in means.groupby("Context_Modality", sort=False):
            plot = context_rows.nlargest(20, "Mean_Attention_Over_Heads").sort_values(
                "Mean_Attention_Over_Heads"
            )
            fig, ax = plt.subplots(figsize=(10, 6))
            for label, label_rows in plot.groupby("True_Label", sort=True):
                ax.barh(
                    label_rows["Context_Token"].astype(str) + (" | benign" if int(label) == 0 else " | pathogenic"),
                    label_rows["Mean_Attention_Over_Heads"],
                    label="Benign" if int(label) == 0 else "Pathogenic",
                    alpha=0.75,
                )
            ax.set_xlabel("Mean attention weight across selected samples and heads")
            ax.set_title(f"Attention query {query_token} → {context_modality} | {PLOT_DATASET}, seed {PLOT_SEED}")
            ax.legend()
            fig.tight_layout()
            safe_query = re.sub(r"[^A-Za-z0-9_.-]+", "_", query_token)
            fig.savefig(OUTPUT_ROOT / f"attention_summary_{safe_query}_{context_modality}_{PLOT_DATASET}_seed{PLOT_SEED}.png", dpi=160)
            plt.show()

# Raw per-layer/head/query/key maps; the six panels are split by context modality.
if not attention_plot_rows.empty:
    selected_variant = str(attention_plot_rows["Variant_ID"].iloc[0])
    safe_variant = re.sub(r"[^A-Za-z0-9_.-]+", "_", selected_variant).strip("_") or "sample"
    map_path = OUTPUT_ROOT / "attention_maps" / PLOT_DATASET / f"seed_{PLOT_SEED}" / f"{safe_variant}.npz"
    if map_path.is_file():
        from matplotlib.colors import PowerNorm
        pairs = [
            ("dna", "protein"), ("dna", "bio_geometry"),
            ("protein", "dna"), ("protein", "bio_geometry"),
            ("bio_geometry", "dna"), ("bio_geometry", "protein"),
        ]
        positions = list(range(-int(MODEL_CONFIG["k_radius"]), int(MODEL_CONFIG["k_radius"]) + 1))
        labels = {
            "dna": [f"DNA:{position:+d}" for position in positions],
            "protein": [f"Protein:{position:+d}" for position in positions],
            "bio_geometry": [*map(str, BIO_FEATURE_NAMES), *map(str, GEOM_FEATURES)],
        }
        matrices = {}
        with np.load(map_path) as archive:
            for query_modality, context_modality in pairs:
                key = f"layer_{PLOT_ATTENTION_LAYER}_{query_modality}_to_{context_modality}"
                if key in archive.files:
                    raw = np.asarray(archive[key], dtype=np.float32)
                    matrices[(query_modality, context_modality)] = raw[0].mean(axis=0)
        if matrices:
            all_attention_values = np.concatenate([matrix.ravel() for matrix in matrices.values()])
            # Attention is highly right-skewed; one near-1 value made linear/shared-max plots look black.
            vmax = float(np.nanpercentile(all_attention_values, 99.5))
            norm = PowerNorm(gamma=0.5, vmin=0.0, vmax=max(vmax, 1e-8), clip=True)
            fig, axes = plt.subplots(3, 2, figsize=(18, 15), constrained_layout=True)
            image = None
            for ax, pair in zip(axes.flat, pairs):
                matrix = matrices.get(pair)
                if matrix is None:
                    ax.axis("off")
                    continue
                query_modality, context_modality = pair
                image = ax.imshow(matrix, aspect="auto", interpolation="nearest", cmap="magma", norm=norm)
                query_labels = labels[query_modality]
                context_labels = labels[context_modality]
                if len(query_labels) != matrix.shape[0]:
                    query_labels = [f"{query_modality}:{i}" for i in range(matrix.shape[0])]
                if len(context_labels) != matrix.shape[1]:
                    context_labels = [f"{context_modality}:{i}" for i in range(matrix.shape[1])]
                ax.set_title(f"{query_modality} queries → {context_modality} keys")
                ax.set_xlabel("Context/key token")
                ax.set_ylabel("Query token")
                xticks = np.linspace(0, len(context_labels) - 1, min(10, len(context_labels)), dtype=int)
                yticks = np.linspace(0, len(query_labels) - 1, min(10, len(query_labels)), dtype=int)
                ax.set_xticks(xticks, [context_labels[i] for i in xticks], rotation=65, ha="right", fontsize=7)
                ax.set_yticks(yticks, [query_labels[i] for i in yticks], fontsize=7)
            if image is not None:
                fig.colorbar(image, ax=list(axes.flat), shrink=0.8, label="Attention weight (head mean; shared sqrt scale, clipped at 99.5th percentile)")
            fig.suptitle(
                f"Cross-attention maps | {PLOT_DATASET} | seed {PLOT_SEED} | "
                f"{selected_variant} | layer {PLOT_ATTENTION_LAYER}\n"
                "Attention is a routing weight over context, not a signed feature contribution."
            )
            fig.savefig(OUTPUT_ROOT / f"attention_heatmap_{PLOT_DATASET}_seed{PLOT_SEED}_layer{PLOT_ATTENTION_LAYER}_{safe_variant}.png", dpi=170, bbox_inches="tight")
            plt.show()
    else:
        print("No raw attention map file:", map_path)

print("Plots and result tables are under:", OUTPUT_ROOT)
# SHAP comparisons use the selected SHAP samples only; do not read as population estimates.
shap_modality_frames = []
if not SHAP_EXACT_GROUPS.empty:
    exact_plot = SHAP_EXACT_GROUPS[SHAP_EXACT_GROUPS["Dataset"].eq(PLOT_DATASET)]
    if PLOT_SEED is not None:
        exact_plot = exact_plot[pd.to_numeric(exact_plot["Seed"], errors="coerce").eq(int(PLOT_SEED))]
    exact_plot = exact_plot.groupby("Modality")["SHAP_Value_Logit"].apply(
        lambda values: values.abs().mean()
    ).rename("Mean_Absolute_SHAP").reset_index()
    exact_plot["Explainer"] = "Exact (4 modality groups)"
    shap_modality_frames.append(exact_plot)
for explainer_name, modality_frame in (
    ("Gradient", SHAP_GRADIENT_MODALITIES),
    ("Deep", SHAP_DEEP_MODALITIES),
):
    if modality_frame.empty:
        continue
    subset = modality_frame[modality_frame["Dataset"].eq(PLOT_DATASET)]
    if PLOT_SEED is not None:
        subset = subset[pd.to_numeric(subset["Seed"], errors="coerce").eq(int(PLOT_SEED))]
    subset = subset.groupby("Modality")["Absolute_Attribution_Sum"].mean().rename(
        "Mean_Absolute_SHAP"
    ).reset_index()
    subset["Explainer"] = explainer_name
    shap_modality_frames.append(subset)
if shap_modality_frames:
    comparison = pd.concat(shap_modality_frames, ignore_index=True)
    pivot = comparison.pivot(index="Modality", columns="Explainer", values="Mean_Absolute_SHAP").fillna(0)
    ax = pivot.plot.bar(figsize=(10, 5))
    ax.set_ylabel("Mean absolute SHAP value (pathogenic logit units)")
    ax.set_title(f"SHAP modality comparison | {PLOT_DATASET} | seed {PLOT_SEED} — Exact uses a single grouped baseline; Gradient/Deep use sampled train3 backgrounds.")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / f"shap_modality_comparison_{PLOT_DATASET}_seed{PLOT_SEED}.png", dpi=160)
    plt.show()

shap_position_frames = []
for explainer_name, position_frame in (
    ("Gradient", SHAP_GRADIENT_POSITIONS),
    ("Deep", SHAP_DEEP_POSITIONS),
):
    if position_frame.empty:
        continue
    subset = position_frame[
        position_frame["Dataset"].eq(PLOT_DATASET)
        & position_frame["Allele"].eq("Alt")
    ]
    if PLOT_SEED is not None:
        subset = subset[pd.to_numeric(subset["Seed"], errors="coerce").eq(int(PLOT_SEED))]
    subset = subset.groupby(["Modality", "Relative_Token_Position"])[
        "Absolute_Attribution"
    ].mean().reset_index()
    subset["Explainer"] = explainer_name
    shap_position_frames.append(subset)
if shap_position_frames:
    positions = pd.concat(shap_position_frames, ignore_index=True)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
    for ax, modality in zip(axes, ("DNA", "Protein")):
        part = positions[positions["Modality"].eq(modality)]
        for explainer_name, group in part.groupby("Explainer"):
            curve = group.groupby("Relative_Token_Position")["Absolute_Attribution"].mean()
            ax.plot(curve.index, curve.values, marker="o", markersize=3, label=explainer_name)
        ax.axvline(0, color="black", linestyle="--", linewidth=1)
        ax.set_title(f"{modality} alt-token SHAP")
        ax.set_xlabel("Relative token position; variant at 0")
        ax.grid(alpha=0.25)
        ax.legend()
    axes[0].set_ylabel("Mean absolute attribution")
    fig.suptitle(f"Gradient/Deep SHAP by token position | {PLOT_DATASET} | seed {PLOT_SEED}")
    fig.tight_layout()
    fig.savefig(OUTPUT_ROOT / f"shap_token_positions_{PLOT_DATASET}_seed{PLOT_SEED}.png", dpi=160)
    plt.show()


# Detailed plots and matched comparisons across IG, occlusion, attention, and SHAP.
from core.module06_architecture_experiments.cross_attention_explainability_plots import (
    create_explainability_comparison,
)
INTERPRETABILITY_COMPARISON = create_explainability_comparison(
    output_root=OUTPUT_ROOT,
    dataset=PLOT_DATASET,
    seed=PLOT_SEED,
    label=PLOT_LABEL,
    mode=PLOT_MODE,
    top_features=PLOT_TOP_FEATURES,
    ig_modalities=IG_MODALITIES,
    ig_features=IG_FEATURES,
    ig_positions=IG_POSITIONS,
    occlusion_individual=OCCLUSION_INDIVIDUAL,
    attention_summary=ATTENTION_SUMMARY,
    shap_exact_groups=SHAP_EXACT_GROUPS,
    shap_gradient_modalities=SHAP_GRADIENT_MODALITIES,
    shap_gradient_features=SHAP_GRADIENT_FEATURES,
    shap_gradient_positions=SHAP_GRADIENT_POSITIONS,
    shap_gradient_completeness=SHAP_GRADIENT_COMPLETENESS,
    shap_deep_modalities=SHAP_DEEP_MODALITIES,
    shap_deep_features=SHAP_DEEP_FEATURES,
    shap_deep_positions=SHAP_DEEP_POSITIONS,
    shap_deep_completeness=SHAP_DEEP_COMPLETENESS,
)
display(INTERPRETABILITY_COMPARISON["modality_method_agreement"])
display(INTERPRETABILITY_COMPARISON["feature_method_agreement"])
display(INTERPRETABILITY_COMPARISON["token_method_agreement"])
display(INTERPRETABILITY_COMPARISON["seed_stability"].head(40))
